OUTLIER DETECTION USING Z-SCORE AND INTERQUARTILE RANGE

In [110]:
import mysql.connector
import configparser
import pandas as pd
from scipy.stats import iqr
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
#Location of the ini file
config = configparser.ConfigParser()
config.read('C:\\Users\\USER\\.spyder-py3\\erfpPROD.ini')

In [111]:
host=config['erfpPROD']['host']
user=config['erfpPROD']['user']
pwd=config['erfpPROD']['pwd']
database=config['erfpPROD']['database']

In [112]:
conn = mysql.connector.connect(
          host=host,
          user=user,
          passwd=pwd,
          database=database)
cursor = conn.cursor()

In [113]:
import_query_location = 'C:\\Users\\USER\\Documents\\Green_score\\5_CERTIFIERS_BM_DATA.sql' 
import_query_open = open(import_query_location, 'r', encoding="utf8")
import_query_read = import_query_open.read()

cursor.execute(import_query_read)

QUERY = cursor.fetchall()
print('Total Row(s):', cursor.rowcount)

# Importing data into a DataFrame
import pandas as pd
erfp_df = pd.DataFrame()
a=[]
for row in QUERY:
        a.append(row)

erfp_df = pd.DataFrame(a)
df_col_names =  [i[0] for i in cursor.description]
erfp_df.columns = df_col_names

In [114]:
erfp_df.shape

In [115]:
erfp_df.head()

In [116]:
erfp_df.MEAS_CARBON_AMT_LBS = pd.to_numeric(erfp_df.MEAS_CARBON_AMT_LBS)
erfp_df.MEAS_ENERGY_AMT_BTU = pd.to_numeric(erfp_df.MEAS_ENERGY_AMT_BTU)
erfp_df.MEAS_WATER_AMT_GALLONS = pd.to_numeric(erfp_df.MEAS_WATER_AMT_GALLONS)
erfp_df.dtypes

In [117]:
# Word split
import re
def find_between(word, start, end):
  return (word.split(start))[1].split(end)[0]

# Visualising to FIND the outliers

components = ['MEAS_CARBON_AMT_LBS', 'MEAS_ENERGY_AMT_BTU', 'MEAS_WATER_AMT_GALLONS']
for i in components:
    plt.figure(figsize=(10,8))
    plt.subplot(211)
    plt.xlim(erfp_df[i].min()*1.5, erfp_df[i].max()*1.1)
    plt.title("Kernel Distribution - "+ find_between(i, '_', '_') + " amount (line graph)") 
    ax = erfp_df[i].plot(kind='kde').margins(0.1)

    plt.subplot(212)
    plt.xlim(erfp_df[i].min()*1.5, erfp_df[i].max()*1.1)
    plt.title("Kernel Distribution -  "+ find_between(i, '_', '_') + " amount (box plot)") 
    sns.boxplot(x=erfp_df[i])

In [126]:
# Remove any zeros (otherwise we get (-inf)
erfp_df.loc[erfp_df.MEAS_CARBON_AMT_LBS == 0, 'MEAS_CARBON_AMT_LBS'] = np.nan
# erfp_df.loc[erfp_df.MEAS_CARBON_AMT_LBS == 9999, 'MEAS_CARBON_AMT_LBS'] = np.nan
# erfp_df.loc[erfp_df.MEAS_CARBON_AMT_LBS == 9999.99, 'MEAS_CARBON_AMT_LBS'] = np.nan

erfp_df.loc[erfp_df.MEAS_ENERGY_AMT_BTU == 0, 'MEAS_ENERGY_AMT_BTU'] = np.nan
# erfp_df.loc[erfp_df.MEAS_ENERGY_AMT_BTU == 9999, 'MEAS_ENERGY_AMT_BTU'] = np.nan
# erfp_df.loc[erfp_df.MEAS_ENERGY_AMT_BTU == 9999.99, 'MEAS_ENERGY_AMT_BTU'] = np.nan

erfp_df.loc[erfp_df.MEAS_WATER_AMT_GALLONS == 0, 'MEAS_WATER_AMT_GALLONS'] = np.nan
# erfp_df.loc[erfp_df.MEAS_WATER_AMT_GALLONS == 9999, 'MEAS_WATER_AMT_GALLONS'] = np.nan
# erfp_df.loc[erfp_df.MEAS_WATER_AMT_GALLONS == 9999.99, 'MEAS_WATER_AMT_GALLONS'] = np.nan

# Drop NA
erfp_df.dropna(inplace=True)

erfp_df.head()

In [127]:
# Visualising Log Transform
components = ['MEAS_CARBON_AMT_LBS', 'MEAS_ENERGY_AMT_BTU', 'MEAS_WATER_AMT_GALLONS']

for i in components:
    j = 'Log-' + i
    erfp_df[j] = np.log(erfp_df[i])
    plt.figure(figsize=(10,8))
    plt.subplot(211)
    plt.xlim(erfp_df[j].min()*1.1, erfp_df[j].max()*1.1)
    plt.title("Kernel Distribution of Log transfomed values - " + find_between(i, '_', '_') + " amount (line graph)") 
    ax = erfp_df[j].plot(kind='kde')

    plt.subplot(212)
    plt.xlim(erfp_df[j].min()*1.1, erfp_df[j].max()*1.1)
    sns.boxplot(x=erfp_df[j])
    plt.title("Kernel Distribution of Log transfomed values - "+ find_between(i, '_', '_') +" amount (box graph)")

In [128]:
# OUTLIER DETECTION USING Z -SCORE
components = ['MEAS_CARBON_AMT_LBS', 'MEAS_ENERGY_AMT_BTU', 'MEAS_WATER_AMT_GALLONS']
for i in components:
    _mean = erfp_df[i].mean()
    _std = erfp_df[i].std()

    erfp_df[i + '_Zscore'] = (erfp_df[i] - _mean)/(_std)
    erfp_df[i + '_Zscore_abs'] = erfp_df[i + '_Zscore'].apply(np.abs)
erfp_df.head()

In [129]:
components = ['MEAS_CARBON_AMT_LBS', 'MEAS_ENERGY_AMT_BTU', 'MEAS_WATER_AMT_GALLONS']
for i in components:
    plt.subplots(figsize=(10,8))
    erfp_df[i + '_Zscore_abs'].hist(color='slategray')
    plt.title("Standard Normal Distribution fo Z-Score - "+ find_between(i, '_', '_') +" amount", y=1.015, fontsize=22)
    plt.xlabel("z-score", labelpad=14)
    plt.ylabel(i, labelpad=14)

In [130]:
# plt.subplots(figsize=(10,8))
erfp_df.hist(column = ['MEAS_CARBON_AMT_LBS_Zscore_abs', 'MEAS_ENERGY_AMT_BTU_Zscore_abs','MEAS_WATER_AMT_GALLONS_Zscore_abs'], bins=10, sharex=True, sharey=True)
# plt.title("Standard Normal Distribution fo Z-Score  amount", y=1.015, fontsize=22)
# plt.xlabel("z-score", labelpad=14)
# plt.ylabel("components", labelpad=14)

In [131]:
dictionary = {'COMPONENTS':['CARBON', 'ENERGY', 'WATER'],
              'COUNT_Z_SCORE_OUTLIERS': [erfp_df.HOTEL_ID[erfp_df.MEAS_CARBON_AMT_LBS_Zscore_abs > 3].count()
                                   ,erfp_df.HOTEL_ID[erfp_df.MEAS_ENERGY_AMT_BTU_Zscore_abs > 3].count()
                                   ,erfp_df.HOTEL_ID[erfp_df.MEAS_WATER_AMT_GALLONS_Zscore_abs > 3].count()]}
Z_SCORE_OUTLIER = pd.DataFrame(dictionary) 
Z_SCORE_OUTLIER

In [132]:
# IQR

# CARBON
carbon_q1, carbon_q3 = np.percentile(erfp_df.Log_MEAS_CARBON_AMT_LBS,[25,75])
carbon_iqr = carbon_q3 - carbon_q1
CARBON_LB = carbon_q1 -(1.5 * carbon_iqr) 
CARBON_UB = carbon_q3 +(1.5 * carbon_iqr) 

# ENERGY
energy_q1, energy_q3 = np.percentile(erfp_df.Log_MEAS_ENERGY_AMT_BTU,[30,60])
energy_iqr = energy_q3 - energy_q1
ENERGY_LB = energy_q1 -(1.5 * energy_iqr) 
ENERGY_UB = energy_q3 +(1.5 * energy_iqr) 

# WATER
water_q1, water_q3 = np.percentile(erfp_df.Log_MEAS_WATER_AMT_GALLONS,[25,75])
water_iqr = water_q3 - water_q1
WATER_LB = water_q1 -(1.5 * water_iqr) 
WATER_UB = water_q3 +(1.5 * water_iqr) 

In [133]:
components = ['MEAS_CARBON_AMT_LBS', 'MEAS_ENERGY_AMT_BTU', 'MEAS_WATER_AMT_GALLONS']
log_components = [ 'Log-'+i for i in components]

for i in log_components:
    if "CARBON" in i:
        LB = CARBON_LB
        UB = CARBON_UB
    elif "ENERGY" in i:
        LB = ENERGY_LB
        UB = ENERGY_UB
    else:
        LB = WATER_LB
        UB = WATER_UB

    plt.figure(figsize=(10,8))
    plt.subplot(211)
    plt.xlim(erfp_df[i].min()*1.1, erfp_df[i].max()*1.1)
    plt.axvline(x=LB, color='r', ls = '--')
    plt.axvspan(xmin=LB, xmax=erfp_df[i].min()*1.1, facecolor='#2ca02c', alpha=0.5)
    plt.axvline(x=UB, color='r', ls = '--')
    plt.axvspan(xmin=UB, xmax=erfp_df[i].max()*1.1, facecolor='#2ca02c', alpha=0.5)
    plt.title("IQR Outlier detection - "+ find_between(i, '_', '_') +" amount (line graph)") 

    ax = erfp_df[i].plot(kind='kde')

    plt.subplot(212)
    plt.xlim(erfp_df[i].min(), erfp_df[i].max()*1.1)
    sns.boxplot(x=erfp_df[i])
    plt.axvline(x=LB, color='r', ls = '--')
    plt.axvspan(xmin=LB, xmax=erfp_df[i].min()*1.1, facecolor='#2ca02c', alpha=0.5)
    plt.axvline(x=UB, color='r', ls = '--')
    plt.axvspan(xmin=UB, xmax=erfp_df[i].max()*1.1, facecolor='#2ca02c', alpha=0.5)
    plt.title("IQR Outlier detection - "+ find_between(i, '_', '_') +" amount (box plot)") 

In [135]:
outlier = ['CARBON_OUTLIER', 'ENERGY_OUTLIER', 'WATER_OUTLIER']
for i in range(0,3):
    if "CARBON" in log_components[i]:
        LB = CARBON_LB
        UB = CARBON_UB
    elif "ENERGY" in log_components[i]:
        LB = ENERGY_LB
        UB = ENERGY_UB
    else:
        LB = WATER_LB
        UB = WATER_UB
    erfp_df[outlier[i]] = 0
    erfp_df.loc[erfp_df[log_components[i]] < LB, outlier[i]] = 1
    erfp_df.loc[erfp_df[log_components[i]] > UB, outlier[i]] = 1


In [136]:
# CARBON OVERIVEW - BENCHMARK
carbon_bm = erfp_df[erfp_df.CARBON_OUTLIER == 0]
carbon_bm = carbon_bm.sort_values(by=['MEAS_CARBON_AMT_LBS'])
print(np.percentile(carbon_bm.MEAS_CARBON_AMT_LBS,[0,25,75,100]))
low25, high25 = carbon_bm.MEAS_CARBON_AMT_LBS.quantile([0.0,0.25])
q25_min = carbon_bm.MEAS_CARBON_AMT_LBS.loc[(carbon_bm.MEAS_CARBON_AMT_LBS >= low25) & (carbon_bm.MEAS_CARBON_AMT_LBS <= high25)].min()
q25_max = carbon_bm.MEAS_CARBON_AMT_LBS.loc[(carbon_bm.MEAS_CARBON_AMT_LBS >= low25) & (carbon_bm.MEAS_CARBON_AMT_LBS <= high25)].max()
q25_mean = carbon_bm.MEAS_CARBON_AMT_LBS.loc[(carbon_bm.MEAS_CARBON_AMT_LBS >= low25) & (carbon_bm.MEAS_CARBON_AMT_LBS <= high25)].mean()
q25_median = carbon_bm.MEAS_CARBON_AMT_LBS.loc[(carbon_bm.MEAS_CARBON_AMT_LBS >= low25) & (carbon_bm.MEAS_CARBON_AMT_LBS <= high25)].median()

low50, high50 = carbon_bm.MEAS_CARBON_AMT_LBS.quantile([0.25,0.5])
q50_min = carbon_bm.MEAS_CARBON_AMT_LBS.loc[(carbon_bm.MEAS_CARBON_AMT_LBS >= low50) & (carbon_bm.MEAS_CARBON_AMT_LBS <= high50)].min()
q50_max = carbon_bm.MEAS_CARBON_AMT_LBS.loc[(carbon_bm.MEAS_CARBON_AMT_LBS >= low50) & (carbon_bm.MEAS_CARBON_AMT_LBS <= high50)].max()
q50_mean = carbon_bm.MEAS_CARBON_AMT_LBS.loc[(carbon_bm.MEAS_CARBON_AMT_LBS >= low50) & (carbon_bm.MEAS_CARBON_AMT_LBS <= high50)].mean()
q50_median = carbon_bm.MEAS_CARBON_AMT_LBS.loc[(carbon_bm.MEAS_CARBON_AMT_LBS >= low50) & (carbon_bm.MEAS_CARBON_AMT_LBS <= high50)].median()

low75, high75 = carbon_bm.MEAS_CARBON_AMT_LBS.quantile([0.5,0.75])
q75_min = carbon_bm.MEAS_CARBON_AMT_LBS.loc[(carbon_bm.MEAS_CARBON_AMT_LBS >= low75) & (carbon_bm.MEAS_CARBON_AMT_LBS <= high75)].min()
q75_max = carbon_bm.MEAS_CARBON_AMT_LBS.loc[(carbon_bm.MEAS_CARBON_AMT_LBS >= low75) & (carbon_bm.MEAS_CARBON_AMT_LBS <= high75)].max()
q75_mean = carbon_bm.MEAS_CARBON_AMT_LBS.loc[(carbon_bm.MEAS_CARBON_AMT_LBS >= low75) & (carbon_bm.MEAS_CARBON_AMT_LBS <= high75)].mean()
q75_median = carbon_bm.MEAS_CARBON_AMT_LBS.loc[(carbon_bm.MEAS_CARBON_AMT_LBS >= low75) & (carbon_bm.MEAS_CARBON_AMT_LBS <= high75)].median()

low100, high100 = carbon_bm.MEAS_CARBON_AMT_LBS.quantile([0.75,1])
q100_min = carbon_bm.MEAS_CARBON_AMT_LBS.loc[(carbon_bm.MEAS_CARBON_AMT_LBS >= low100) & (carbon_bm.MEAS_CARBON_AMT_LBS < high100)].min()
q100_max = carbon_bm.MEAS_CARBON_AMT_LBS.loc[(carbon_bm.MEAS_CARBON_AMT_LBS >= low100) & (carbon_bm.MEAS_CARBON_AMT_LBS < high100)].max()
q100_mean = carbon_bm.MEAS_CARBON_AMT_LBS.loc[(carbon_bm.MEAS_CARBON_AMT_LBS >= low100) & (carbon_bm.MEAS_CARBON_AMT_LBS < high100)].mean()
q100_median = carbon_bm.MEAS_CARBON_AMT_LBS.loc[(carbon_bm.MEAS_CARBON_AMT_LBS >= low100) & (carbon_bm.MEAS_CARBON_AMT_LBS < high100)].median()


carbon_overview = {'ALL': [round(carbon_bm.MEAS_CARBON_AMT_LBS.min(), 3), round(carbon_bm.MEAS_CARBON_AMT_LBS.max(),3), round(carbon_bm.MEAS_CARBON_AMT_LBS.mean(),3),round(carbon_bm.MEAS_CARBON_AMT_LBS.median(),3)],
            'Q25': [round(q25_min, 3), round(q25_max,3), round(q25_mean,3),round(q25_median,3)],
            'Q50': [round(q50_min, 3), round(q50_max,3), round(q50_mean,3),round(q50_median,3)],
            'Q75': [round(q75_min, 3), round(q75_max,3), round(q75_mean,3),round(q75_median,3)],
            'Q100': [round(q100_min, 3), round(q100_max,3), round(q100_mean,3),round(q100_median,3)]
           }
carbon_overview

In [137]:
# ENERGY OVERIVEW - BENCHMARK
energy_bm = erfp_df[erfp_df.ENERGY_OUTLIER == 0]
energy_bm = energy_bm.sort_values(by=['MEAS_ENERGY_AMT_BTU'])
print(np.percentile(energy_bm.MEAS_ENERGY_AMT_BTU,[0,25,75,100]))
low25, high25 = energy_bm.MEAS_ENERGY_AMT_BTU.quantile([0.0,0.25])
q25_min = energy_bm.MEAS_ENERGY_AMT_BTU.loc[(energy_bm.MEAS_ENERGY_AMT_BTU >= low25) & (energy_bm.MEAS_ENERGY_AMT_BTU <= high25)].min()
q25_max = energy_bm.MEAS_ENERGY_AMT_BTU.loc[(energy_bm.MEAS_ENERGY_AMT_BTU >= low25) & (energy_bm.MEAS_ENERGY_AMT_BTU <= high25)].max()
q25_mean = energy_bm.MEAS_ENERGY_AMT_BTU.loc[(energy_bm.MEAS_ENERGY_AMT_BTU >= low25) & (energy_bm.MEAS_ENERGY_AMT_BTU <= high25)].mean()
q25_median = energy_bm.MEAS_ENERGY_AMT_BTU.loc[(energy_bm.MEAS_ENERGY_AMT_BTU >= low25) & (energy_bm.MEAS_ENERGY_AMT_BTU <= high25)].median()

low50, high50 = energy_bm.MEAS_ENERGY_AMT_BTU.quantile([0.25,0.5])
q50_min = energy_bm.MEAS_ENERGY_AMT_BTU.loc[(energy_bm.MEAS_ENERGY_AMT_BTU >= low50) & (energy_bm.MEAS_ENERGY_AMT_BTU <= high50)].min()
q50_max = energy_bm.MEAS_ENERGY_AMT_BTU.loc[(energy_bm.MEAS_ENERGY_AMT_BTU >= low50) & (energy_bm.MEAS_ENERGY_AMT_BTU <= high50)].max()
q50_mean = energy_bm.MEAS_ENERGY_AMT_BTU.loc[(energy_bm.MEAS_ENERGY_AMT_BTU >= low50) & (energy_bm.MEAS_ENERGY_AMT_BTU <= high50)].mean()
q50_median = energy_bm.MEAS_ENERGY_AMT_BTU.loc[(energy_bm.MEAS_ENERGY_AMT_BTU >= low50) & (energy_bm.MEAS_ENERGY_AMT_BTU <= high50)].median()

low75, high75 = energy_bm.MEAS_ENERGY_AMT_BTU.quantile([0.5,0.75])
q75_min = energy_bm.MEAS_ENERGY_AMT_BTU.loc[(energy_bm.MEAS_ENERGY_AMT_BTU >= low75) & (energy_bm.MEAS_ENERGY_AMT_BTU <= high75)].min()
q75_max = energy_bm.MEAS_ENERGY_AMT_BTU.loc[(energy_bm.MEAS_ENERGY_AMT_BTU >= low75) & (energy_bm.MEAS_ENERGY_AMT_BTU <= high75)].max()
q75_mean = energy_bm.MEAS_ENERGY_AMT_BTU.loc[(energy_bm.MEAS_ENERGY_AMT_BTU >= low75) & (energy_bm.MEAS_ENERGY_AMT_BTU <= high75)].mean()
q75_median = energy_bm.MEAS_ENERGY_AMT_BTU.loc[(energy_bm.MEAS_ENERGY_AMT_BTU >= low75) & (energy_bm.MEAS_ENERGY_AMT_BTU <= high75)].median()

low100, high100 = energy_bm.MEAS_ENERGY_AMT_BTU.quantile([0.75,1])
q100_min = energy_bm.MEAS_ENERGY_AMT_BTU.loc[(energy_bm.MEAS_ENERGY_AMT_BTU >= low100) & (energy_bm.MEAS_ENERGY_AMT_BTU < high100)].min()
q100_max = energy_bm.MEAS_ENERGY_AMT_BTU.loc[(energy_bm.MEAS_ENERGY_AMT_BTU >= low100) & (energy_bm.MEAS_ENERGY_AMT_BTU < high100)].max()
q100_mean = energy_bm.MEAS_ENERGY_AMT_BTU.loc[(energy_bm.MEAS_ENERGY_AMT_BTU >= low100) & (energy_bm.MEAS_ENERGY_AMT_BTU < high100)].mean()
q100_median = energy_bm.MEAS_ENERGY_AMT_BTU.loc[(energy_bm.MEAS_ENERGY_AMT_BTU >= low100) & (energy_bm.MEAS_ENERGY_AMT_BTU < high100)].median()



energy_overview = {'ALL': [round(energy_bm.MEAS_ENERGY_AMT_BTU.min(), 3), round(energy_bm.MEAS_ENERGY_AMT_BTU.max(),3), round(energy_bm.MEAS_ENERGY_AMT_BTU.mean(),3),round(energy_bm.MEAS_ENERGY_AMT_BTU.median(),3)],
            'Q25': [round(q25_min, 3), round(q25_max,3), round(q25_mean,3),round(q25_median,3)],
            'Q50': [round(q50_min, 3), round(q50_max,3), round(q50_mean,3),round(q50_median,3)],
            'Q75': [round(q75_min, 3), round(q75_max,3), round(q75_mean,3),round(q75_median,3)],
            'Q100': [round(q100_min, 3), round(q100_max,3), round(q100_mean,3),round(q100_median,3)]
           }
energy_overview 

In [138]:
# WATER OVERIVEW - BENCHMARK
water_bm = erfp_df[erfp_df.WATER_OUTLIER == 0]
water_bm = water_bm.sort_values(by=['MEAS_WATER_AMT_GALLONS'])
print(np.percentile(water_bm.MEAS_WATER_AMT_GALLONS,[0,25,75,100]))
low25, high25 = water_bm.MEAS_WATER_AMT_GALLONS.quantile([0.0,0.25])
q25_min = water_bm.MEAS_WATER_AMT_GALLONS.loc[(water_bm.MEAS_WATER_AMT_GALLONS >= low25) & (water_bm.MEAS_WATER_AMT_GALLONS <= high25)].min()
q25_max = water_bm.MEAS_WATER_AMT_GALLONS.loc[(water_bm.MEAS_WATER_AMT_GALLONS >= low25) & (water_bm.MEAS_WATER_AMT_GALLONS <= high25)].max()
q25_mean = water_bm.MEAS_WATER_AMT_GALLONS.loc[(water_bm.MEAS_WATER_AMT_GALLONS >= low25) & (water_bm.MEAS_WATER_AMT_GALLONS <= high25)].mean()
q25_median = water_bm.MEAS_WATER_AMT_GALLONS.loc[(water_bm.MEAS_WATER_AMT_GALLONS >= low25) & (water_bm.MEAS_WATER_AMT_GALLONS <= high25)].median()

low50, high50 = water_bm.MEAS_WATER_AMT_GALLONS.quantile([0.25,0.5])
q50_min = water_bm.MEAS_WATER_AMT_GALLONS.loc[(water_bm.MEAS_WATER_AMT_GALLONS >= low50) & (water_bm.MEAS_WATER_AMT_GALLONS <= high50)].min()
q50_max = water_bm.MEAS_WATER_AMT_GALLONS.loc[(water_bm.MEAS_WATER_AMT_GALLONS >= low50) & (water_bm.MEAS_WATER_AMT_GALLONS <= high50)].max()
q50_mean = water_bm.MEAS_WATER_AMT_GALLONS.loc[(water_bm.MEAS_WATER_AMT_GALLONS >= low50) & (water_bm.MEAS_WATER_AMT_GALLONS <= high50)].mean()
q50_median = water_bm.MEAS_WATER_AMT_GALLONS.loc[(water_bm.MEAS_WATER_AMT_GALLONS >= low50) & (water_bm.MEAS_WATER_AMT_GALLONS <= high50)].median()

low75, high75 = water_bm.MEAS_WATER_AMT_GALLONS.quantile([0.5,0.75])
q75_min = water_bm.MEAS_WATER_AMT_GALLONS.loc[(water_bm.MEAS_WATER_AMT_GALLONS >= low75) & (water_bm.MEAS_WATER_AMT_GALLONS <= high75)].min()
q75_max = water_bm.MEAS_WATER_AMT_GALLONS.loc[(water_bm.MEAS_WATER_AMT_GALLONS >= low75) & (water_bm.MEAS_WATER_AMT_GALLONS <= high75)].max()
q75_mean = water_bm.MEAS_WATER_AMT_GALLONS.loc[(water_bm.MEAS_WATER_AMT_GALLONS >= low75) & (water_bm.MEAS_WATER_AMT_GALLONS <= high75)].mean()
q75_median = water_bm.MEAS_WATER_AMT_GALLONS.loc[(water_bm.MEAS_WATER_AMT_GALLONS >= low75) & (water_bm.MEAS_WATER_AMT_GALLONS <= high75)].median()

low100, high100 = water_bm.MEAS_WATER_AMT_GALLONS.quantile([0.75,1])
q100_min = water_bm.MEAS_WATER_AMT_GALLONS.loc[(water_bm.MEAS_WATER_AMT_GALLONS >= low100) & (water_bm.MEAS_WATER_AMT_GALLONS < high100)].min()
q100_max = water_bm.MEAS_WATER_AMT_GALLONS.loc[(water_bm.MEAS_WATER_AMT_GALLONS >= low100) & (water_bm.MEAS_WATER_AMT_GALLONS < high100)].max()
q100_mean = water_bm.MEAS_WATER_AMT_GALLONS.loc[(water_bm.MEAS_WATER_AMT_GALLONS >= low100) & (water_bm.MEAS_WATER_AMT_GALLONS < high100)].mean()
q100_median = water_bm.MEAS_WATER_AMT_GALLONS.loc[(water_bm.MEAS_WATER_AMT_GALLONS >= low100) & (water_bm.MEAS_WATER_AMT_GALLONS < high100)].median()


water_overview = {'ALL': [round(water_bm.MEAS_WATER_AMT_GALLONS.min(), 3), round(water_bm.MEAS_WATER_AMT_GALLONS.max(),3), round(water_bm.MEAS_WATER_AMT_GALLONS.mean(),3),round(water_bm.MEAS_WATER_AMT_GALLONS.median(),3)],
            'Q25': [round(q25_min, 3), round(q25_max,3), round(q25_mean,3),round(q25_median,3)],
            'Q50': [round(q50_min, 3), round(q50_max,3), round(q50_mean,3),round(q50_median,3)],
            'Q75': [round(q75_min, 3), round(q75_max,3), round(q75_mean,3),round(q75_median,3)],
            'Q100': [round(q100_min, 3), round(q100_max,3), round(q100_mean,3),round(q100_median,3)]
           }
water_overview

In [142]:
carbon_overview_df = pd.DataFrame.from_dict(carbon_overview, orient='index', columns= ['MIN', 'MAX', 'MEAN', 'MEDIAN'])
carbon_overview_df['SOURCE'] = 'CARBON'
energy_overview_df = pd.DataFrame.from_dict(energy_overview, orient='index', columns= ['MIN', 'MAX', 'MEAN', 'MEDIAN'])
energy_overview_df['SOURCE'] = 'ENERGY'
water_overview_df = pd.DataFrame.from_dict(water_overview, orient='index', columns= ['MIN', 'MAX', 'MEAN', 'MEDIAN'])
water_overview_df['SOURCE'] = 'WATER'
overview_df = pd.concat([carbon_overview_df, energy_overview_df, water_overview_df], axis=0)
overview_df

In [143]:
# OUTLIER

import datetime as d
file = "C:\\Users\\USER\\Documents\\Green_score\\5_CERTIFIER_BM.xlsx" 
with pd.ExcelWriter(file) as writer: 
    overview_df.to_excel(writer, sheet_name='OVERVIEW', header=True, encoding='utf-8', index=True)
    erfp_df[erfp_df.CARBON_OUTLIER == 1].to_excel(writer, sheet_name='CARBON_OUTLIER', header=True, encoding='utf-8', index=False, freeze_panes=(1,1))
    erfp_df[erfp_df.CARBON_OUTLIER == 0].to_excel(writer, sheet_name='CARBON_BENCHMARK', header=True, encoding='utf-8', index=False, freeze_panes=(1,1))
    erfp_df[erfp_df.ENERGY_OUTLIER == 1].to_excel(writer, sheet_name='ENERGY_OUTLIER', header=True, encoding='utf-8', index=False, freeze_panes=(1,1))
    erfp_df[erfp_df.ENERGY_OUTLIER == 0].to_excel(writer, sheet_name='ENERGY_BENCHMARK', header=True, encoding='utf-8', index=False, freeze_panes=(1,1))
    erfp_df[erfp_df.WATER_OUTLIER == 1].to_excel(writer, sheet_name='WATER_OUTLIER', header=True, encoding='utf-8', index=False, freeze_panes=(1,1))
    erfp_df[erfp_df.WATER_OUTLIER == 0].to_excel(writer, sheet_name='WATER_BENCHMARK', header=True, encoding='utf-8', index=False, freeze_panes=(1,1))

THE END
                                              